# TrackID3x3 — Kaggle GPU compatibility baseline

Status before execution: **READY_FOR_KAGGLE**. No successful Kaggle run is claimed by this file.

Enable a GPU and Internet in Kaggle Settings, then **Restart Session → Run All**.
For offline use, attach the original four tutorial files, `yolo11x.pt`, default EasyOCR weights,
and any missing package wheels; set `ALLOW_DOWNLOADS=False` and the input directories in the first cell.
Do not change models, thresholds, roster, filters or interpolation to match historical scores.

Outputs: `/kaggle/working/trackid3x3_runs/<run-id>/`. Videos are optional (`SAVE_VIDEOS=False` by default).
The original remains `TrackID3x3_Challenge_Tutorial.ipynb`, SHA-256
`33da120728e1ef5884ef001a8c30da3b0c183c165626b91c1075f1be7e9c1c00`.

The source notebook hash and a source-only replay snapshot are saved at completion.
Runtime code history captures code edits; UI markdown edits require exporting the notebook separately.


In [ ]:
# TrackID3x3 Kaggle cell: setup
"""Embedded by build_kaggle_notebook.py; the generated notebook is standalone."""
import contextlib
import hashlib
import importlib
import importlib.util
import importlib.metadata as metadata
import io
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import time
import traceback
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

# ENVIRONMENT / PATH: edit these only when using mounted offline inputs.
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd() / 'trackid3x3_work'
INPUT_DIR = Path('/kaggle/input')
DATA_INPUT_DIR = None     # e.g. Path('/kaggle/input/trackid3x3-tutorial')
MODEL_INPUT_DIR = None    # optional directory containing original YOLO/EasyOCR weights
WHEEL_DIR = None          # optional directory of compatible wheels for offline missing packages
ALLOW_DOWNLOADS = True   # False: no data/model/PyPI network requests
SAVE_VIDEOS = False      # optional visualization only; metrics/predictions are unchanged
PROBE_FRAMES = 12         # isolated real tracking format check; never used as baseline predictions
DATA_DIR = WORK_DIR / 'trackid3x3_tutorial_data'
OUTPUT_DIR = WORK_DIR / 'trackid3x3_runs'
RUN_ID = datetime.now(ZoneInfo('Asia/Ho_Chi_Minh')).strftime('%Y%m%dT%H%M%S%f') + '_baseline_kaggle'
RUN_DIR = OUTPUT_DIR / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
TEMP_DIR = RUN_DIR / 'tmp'
TEMP_DIR.mkdir()
OCR_MODEL_DIR = RUN_DIR / 'models' / 'easyocr'
OCR_MODEL_DIR.mkdir(parents=True)
IS_KAGGLE = INPUT_DIR.is_dir() and Path('/kaggle/working').is_dir()
ORIGINAL_NOTEBOOK_SHA256 = '33da120728e1ef5884ef001a8c30da3b0c183c165626b91c1075f1be7e9c1c00'
# Builder injects source-only notebook structure here, to capture exact executed code later.
NOTEBOOK_LAYOUT = {'cells': [{'cell_type': 'markdown', 'id': 'intro', 'metadata': {}, 'source': ['# TrackID3x3 — Kaggle GPU compatibility baseline\n', '\n', 'Status before execution: **READY_FOR_KAGGLE**. No successful Kaggle run is claimed by this file.\n', '\n', 'Enable a GPU and Internet in Kaggle Settings, then **Restart Session → Run All**.\n', 'For offline use, attach the original four tutorial files, `yolo11x.pt`, default EasyOCR weights,\n', 'and any missing package wheels; set `ALLOW_DOWNLOADS=False` and the input directories in the first cell.\n', 'Do not change models, thresholds, roster, filters or interpolation to match historical scores.\n', '\n', 'Outputs: `/kaggle/working/trackid3x3_runs/<run-id>/`. Videos are optional (`SAVE_VIDEOS=False` by default).\n', 'The original remains `TrackID3x3_Challenge_Tutorial.ipynb`, SHA-256\n', '`33da120728e1ef5884ef001a8c30da3b0c183c165626b91c1075f1be7e9c1c00`.\n', '\n', 'The source notebook hash and a source-only replay snapshot are saved at completion.\n', 'Runtime code history captures code edits; UI markdown edits require exporting the notebook separately.\n']}, {'cell_type': 'code', 'id': 'setup', 'metadata': {'change_categories': ['ENVIRONMENT', 'PATH', 'DEPENDENCY', 'OBSERVABILITY']}, 'execution_count': None, 'outputs': [], 'source': []}, {'cell_type': 'code', 'id': 'environment', 'metadata': {'original_cell': 3, 'change_categories': ['ENVIRONMENT', 'DEPENDENCY', 'OBSERVABILITY']}, 'execution_count': None, 'outputs': [], 'source': []}, {'cell_type': 'code', 'id': 'data', 'metadata': {'original_cell': 5, 'change_categories': ['PATH', 'DEPENDENCY', 'OBSERVABILITY']}, 'execution_count': None, 'outputs': [], 'source': []}, {'cell_type': 'code', 'id': 'label-probe', 'metadata': {'change_categories': ['LIBRARY-COMPATIBILITY', 'OBSERVABILITY']}, 'execution_count': None, 'outputs': [], 'source': []}, {'cell_type': 'code', 'id': 'tracking', 'metadata': {'original_cell': 7, 'change_categories': ['PATH', 'VISUALIZATION', 'OBSERVABILITY']}, 'execution_count': None, 'outputs': [], 'source': []}, {'cell_type': 'code', 'id': 'mot', 'metadata': {'original_cell': 9, 'change_categories': ['OBSERVABILITY']}, 'execution_count': None, 'outputs': [], 'source': []}, {'cell_type': 'code', 'id': 'homography', 'metadata': {'original_cell': 11, 'change_categories': ['OBSERVABILITY']}, 'execution_count': None, 'outputs': [], 'source': []}, {'cell_type': 'code', 'id': 'court-filtering', 'metadata': {'original_cell': 15, 'change_categories': ['OBSERVABILITY']}, 'execution_count': None, 'outputs': [], 'source': []}, {'cell_type': 'code', 'id': 'ocr', 'metadata': {'original_cell': 17, 'change_categories': ['PATH', 'ENVIRONMENT', 'OBSERVABILITY']}, 'execution_count': None, 'outputs': [], 'source': []}, {'cell_type': 'code', 'id': 'teams', 'metadata': {'original_cell': 17, 'change_categories': ['OBSERVABILITY']}, 'execution_count': None, 'outputs': [], 'source': []}, {'cell_type': 'code', 'id': 'identity', 'metadata': {'original_cell': 17, 'change_categories': ['OBSERVABILITY']}, 'execution_count': None, 'outputs': [], 'source': []}, {'cell_type': 'code', 'id': 'prediction', 'metadata': {'original_cell': 17, 'change_categories': ['OBSERVABILITY']}, 'execution_count': None, 'outputs': [], 'source': []}, {'cell_type': 'code', 'id': 'evaluator-definitions', 'metadata': {'original_cell': 21}, 'execution_count': None, 'outputs': [], 'source': []}, {'cell_type': 'code', 'id': 'evaluation', 'metadata': {'original_cell': [23, 24], 'change_categories': ['OBSERVABILITY']}, 'execution_count': None, 'outputs': [], 'source': []}, {'cell_type': 'markdown', 'id': 'optional-video-note', 'metadata': {}, 'source': ['## Optional visualization\n', '\n', 'When `SAVE_VIDEOS=True`, the following cell creates the original tracker video, rectified\n', 'court image, GT/pred minimaps, and combined video. It runs **after persisted evaluation**.\n', 'Only Colab display and destination paths are replaced. The original minimap time handling\n', 'and drawing parameters are preserved. No metric depends on this cell.\n']}, {'cell_type': 'code', 'id': 'optional-videos', 'metadata': {'original_cell': [9, 13, 26, 28, 30], 'change_categories': ['PATH', 'VISUALIZATION', 'OBSERVABILITY']}, 'execution_count': None, 'outputs': [], 'source': []}, {'cell_type': 'code', 'id': 'finish', 'metadata': {'change_categories': ['OBSERVABILITY']}, 'execution_count': None, 'outputs': [], 'source': []}], 'metadata': {'kernelspec': {'display_name': 'Python 3', 'language': 'python', 'name': 'python3'}, 'language_info': {'name': 'python'}, 'accelerator': 'GPU'}, 'nbformat': 4, 'nbformat_minor': 5}


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def save_json(name, obj):
    (RUN_DIR / name).write_text(json.dumps(obj, indent=2, allow_nan=False, default=str) + '\n')


RUN_STATE = dict(status='RUNNING', platform='Kaggle' if IS_KAGGLE else 'other',
                 original_notebook_sha256=ORIGINAL_NOTEBOOK_SHA256,
                 started_utc=datetime.now(timezone.utc).isoformat(),
                 seeds='unset, as in original notebook', stages=[])
save_json('run_manifest.json', RUN_STATE)
save_json('configuration.json', dict(WORK_DIR=str(WORK_DIR), INPUT_DIR=str(INPUT_DIR),
          DATA_INPUT_DIR=str(DATA_INPUT_DIR) if DATA_INPUT_DIR else None,
          MODEL_INPUT_DIR=str(MODEL_INPUT_DIR) if MODEL_INPUT_DIR else None,
          WHEEL_DIR=str(WHEEL_DIR) if WHEEL_DIR else None,
          DATA_DIR=str(DATA_DIR), OUTPUT_DIR=str(OUTPUT_DIR), RUN_DIR=str(RUN_DIR),
          TEMP_DIR=str(TEMP_DIR), ALLOW_DOWNLOADS=ALLOW_DOWNLOADS, SAVE_VIDEOS=SAVE_VIDEOS,
          PROBE_FRAMES=PROBE_FRAMES, seed_policy='unset; original behavior'))


class Tee:
    def __init__(self, original, logfile):
        self.original, self.logfile = original, logfile
    def write(self, value):
        self.logfile.write(value)
        self.logfile.flush()
        return self.original.write(value)
    def flush(self):
        self.logfile.flush()
        self.original.flush()
    def __getattr__(self, key):
        return getattr(self.original, key)


@contextlib.contextmanager
def baseline_stage(number, name):
    start = time.perf_counter()
    entry = dict(stage=number, name=name, status='RUNNING')
    RUN_STATE['stages'].append(entry)
    save_json('run_manifest.json', RUN_STATE)
    with (RUN_DIR / 'stages.log').open('a') as log:
        with contextlib.redirect_stdout(Tee(sys.stdout, log)), contextlib.redirect_stderr(Tee(sys.stderr, log)):
            print(f'[{number}] {name}')
            try:
                yield
            except BaseException:
                entry['status'] = 'FAILED'
                RUN_STATE['status'] = 'FAILED'
                (RUN_DIR / 'failure.log').write_text(traceback.format_exc())
                raise
            else:
                entry['status'] = 'COMPLETE'
            finally:
                entry['elapsed_seconds'] = time.perf_counter() - start
                save_json('run_manifest.json', RUN_STATE)
                print(f'[{number}] {entry["status"]} ({entry["elapsed_seconds"]:.2f}s)')


def choose_unique(root, filename):
    root = Path(root)
    matches = sorted(root.rglob(filename)) if root.is_dir() else []
    if len(matches) > 1:
        raise RuntimeError(f'Ambiguous {filename} in {root}: {matches}. Set a specific input directory.')
    return matches[0] if matches else None


def install_missing_dependencies():
    # DEPENDENCY: retain the Kaggle numeric and PyTorch/CUDA stack.
    from packaging.version import Version
    required = [('ultralytics', 'ultralytics', '8.3.0'), ('easyocr', 'easyocr', None),
                ('ndjson', 'ndjson', None), ('lap', 'lap', '0.5.12')]
    if ALLOW_DOWNLOADS:
        required.append(('gdown', 'gdown', None))
    if importlib.util.find_spec('cv2') is None:
        required.append(('opencv-python', 'cv2', None))
    for name in ('torch', 'torchvision', 'numpy', 'pandas', 'scipy', 'IPython', 'packaging'):
        if importlib.util.find_spec(name) is None:
            raise RuntimeError(f'Kaggle runtime is missing {name}; use the standard GPU image. Core stack is not auto-upgraded.')
    constraints = []
    protected = {}
    for dist in metadata.distributions():
        name = dist.metadata.get('Name', '').lower().replace('_', '-')
        if name in {'torch', 'torchvision', 'torchaudio', 'numpy', 'pandas', 'scipy',
                    'opencv-python', 'opencv-python-headless', 'opencv-contrib-python', 'triton'} or name.startswith('nvidia-'):
            protected[name] = dist.version
            constraints.append(f'{name}=={dist.version}')
    constraint_path = RUN_DIR / 'protected-runtime-constraints.txt'
    constraint_path.write_text('\n'.join(sorted(set(constraints))) + '\n')
    missing = []
    for dist, module, minimum in required:
        try:
            version = metadata.version(dist)
        except metadata.PackageNotFoundError:
            version = None
        if version is None or (minimum and Version(version) < Version(minimum)):
            missing.append(dist + (f'>={minimum}' if minimum else ''))
    save_json('dependency_plan.json', dict(install=missing, protected=protected))
    if missing:
        command = [sys.executable, '-m', 'pip', 'install', '--constraint', str(constraint_path)]
        if WHEEL_DIR is not None:
            command += ['--find-links', str(WHEEL_DIR)]
        if not ALLOW_DOWNLOADS:
            if WHEEL_DIR is None:
                raise RuntimeError(f'Missing packages {missing}. Enable Internet or add wheels and set WHEEL_DIR.')
            command += ['--no-index']
        command += missing
        process = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        (RUN_DIR / 'dependency_install.log').write_text(process.stdout)
        if process.returncode:
            raise RuntimeError('Dependency install failed under protected runtime constraints; see dependency_install.log. No automatic core-stack upgrade.')
        importlib.invalidate_caches()
    else:
        (RUN_DIR / 'dependency_install.log').write_text('No dependency installation needed.\n')
    for name, version in protected.items():
        if metadata.version(name) != version:
            raise RuntimeError(f'Protected package changed: {name}')
    # Import failures are recorded; do not speculatively replace installed packages.
    for _, module, _ in required:
        importlib.import_module(module)


def save_environment():
    import torch
    packages = {}
    for name in ('torch', 'torchvision', 'ultralytics', 'opencv-python', 'opencv-python-headless',
                 'easyocr', 'numpy', 'pandas', 'scipy', 'ndjson', 'gdown', 'lap'):
        try:
            packages[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            packages[name] = None
    env = dict(python=sys.version, executable=sys.executable, platform=platform.platform(),
               packages=packages, torch_version=torch.__version__, cuda_available=torch.cuda.is_available(),
               cuda_version=torch.version.cuda, cudnn_version=torch.backends.cudnn.version(),
               gpu_name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
    save_json('environment.json', env)
    freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True)
    (RUN_DIR / 'pip-freeze.txt').write_text(freeze.stdout + freeze.stderr)
    if freeze.returncode:
        raise RuntimeError('pip freeze failed; see pip-freeze.txt')
    print('torch version:', env['torch_version'])
    print('CUDA available:', env['cuda_available'])
    print('CUDA version:', env['cuda_version'])
    print('GPU name:', env['gpu_name'])
    print('Packages:', json.dumps(packages))
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA GPU required. On Kaggle enable a GPU accelerator, Restart Session, then Run All. CPU fallback is disabled.')
    torch.empty(1, device='cuda:0')  # verifies device 0 is usable


def resolve_data():
    filenames = ['IMG_0104_1.MOV', 'IMG_0104_1_tutorial_challenge.csv', 'Outdoor.ndjson', 'Outdoor.png']
    input_root = Path(DATA_INPUT_DIR) if DATA_INPUT_DIR is not None else INPUT_DIR
    found = {name: choose_unique(input_root, name) for name in filenames}
    if not all(found.values()):
        if DATA_INPUT_DIR is not None:
            raise FileNotFoundError(f'Incomplete explicit DATA_INPUT_DIR {input_root}: missing {[n for n,p in found.items() if p is None]}')
        # Downloaded cache is checked separately; never mix partial input datasets.
        found = {name: choose_unique(DATA_DIR, name) for name in filenames}
        if not all(found.values()):
            if not ALLOW_DOWNLOADS:
                raise FileNotFoundError('Add the original four tutorial files under /kaggle/input and set DATA_INPUT_DIR. Downloads are disabled.')
            DATA_DIR.mkdir(parents=True, exist_ok=True)
            command = [sys.executable, '-m', 'gdown',
                       'https://drive.google.com/drive/folders/1_yDMy8YVpxyqcKg2_eXN1xz_vg2FeCGu',
                       '--folder', '-O', str(DATA_DIR)]
            process = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            (RUN_DIR / 'data_download.log').write_text(process.stdout)
            if process.returncode:
                raise RuntimeError('Original gdown folder download failed. See data_download.log; add the same original files as a Kaggle Dataset input.')
            found = {name: choose_unique(DATA_DIR, name) for name in filenames}
    if not all(found.values()):
        raise FileNotFoundError(f'Missing original tutorial inputs: {[n for n,p in found.items() if p is None]}')
    known = {'IMG_0104_1_tutorial_challenge.csv': 'a86beb68f5e8351eea3b65b126db2a167b4c2f2dc805715818951c6760f4f2a2',
             'Outdoor.ndjson': '84bedee6b8b499e84bb98dba99b8c36a11c3eb08c60e88c3e7fbc95979b1b9ca'}
    inventory = {}
    for name, path in found.items():
        digest = file_sha256(path)
        inventory[name] = dict(path=str(path), bytes=path.stat().st_size, sha256=digest)
        if name in known and digest != known[name]:
            raise RuntimeError(f'{name} differs from the audited tutorial input; verify the dataset instead of silently substituting it.')
        print(f'{name}: {path}')
    save_json('inputs.json', inventory)
    import cv2
    cap = cv2.VideoCapture(str(found[filenames[0]]))
    ok, _ = cap.read()
    info = dict(width=cap.get(cv2.CAP_PROP_FRAME_WIDTH), height=cap.get(cv2.CAP_PROP_FRAME_HEIGHT),
                frames=cap.get(cv2.CAP_PROP_FRAME_COUNT), fps=cap.get(cv2.CAP_PROP_FPS))
    cap.release()
    save_json('video_properties.json', info)
    if not ok or min(info.values()) <= 0:
        raise RuntimeError('Original tutorial video cannot be decoded; inspect video_properties.json.')
    return tuple(str(found[name]) for name in filenames)


def prepare_models():
    import ultralytics
    from easyocr.config import detection_models, recognition_models
    model_root = Path(MODEL_INPUT_DIR) if MODEL_INPUT_DIR is not None else INPUT_DIR
    weights = choose_unique(model_root, 'yolo11x.pt')
    target = RUN_DIR / 'models' / 'yolo11x.pt'
    if weights is None:
        cached = WORK_DIR / 'yolo11x.pt'
        weights = cached if cached.is_file() else None
    if weights:
        shutil.copy2(weights, target)
    elif ALLOW_DOWNLOADS:
        # Same YOLO11x model; destination is a writable run-local path.
        from ultralytics import YOLO
        YOLO(str(target))
    else:
        raise FileNotFoundError('Offline run needs the original yolo11x.pt in MODEL_INPUT_DIR or /kaggle/input.')
    if not target.is_file():
        raise FileNotFoundError(f'YOLO11x weight download did not produce {target}')
    # Reader(["en"], gpu=True) defaults: craft detector, standard English recognizer.
    required = [detection_models['craft']['filename'], recognition_models['gen2']['english_g2']['filename']]
    for filename in required:
        source = choose_unique(model_root, filename)
        if source is None:
            source = choose_unique(Path.home() / '.EasyOCR' / 'model', filename)
        if source:
            shutil.copy2(source, OCR_MODEL_DIR / filename)
        elif not ALLOW_DOWNLOADS:
            raise FileNotFoundError(f'Offline EasyOCR needs {filename} in MODEL_INPUT_DIR; no alternate OCR model will be selected.')
    yaml_source = Path(ultralytics.__file__).resolve().parent / 'cfg' / 'trackers' / 'bytetrack.yaml'
    if not yaml_source.is_file():
        raise FileNotFoundError(f'Installed Ultralytics ByteTrack YAML missing: {yaml_source}')
    shutil.copy2(yaml_source, RUN_DIR / 'bytetrack.yaml')
    save_json('models_before_inference.json', dict(yolo_path=str(target), yolo_sha256=file_sha256(target),
              bytetrack_source=str(yaml_source), bytetrack_sha256=file_sha256(yaml_source),
              easyocr_default_files=required))
    return str(target)


def capture_notebook_source(history):
    # Hash a persisted source notebook, not a guessed pathname or an output-bearing export.
    # Raw executed code comes from this kernel; markdown/metadata come from the build layout.
    observed = {}
    for raw in history:
        for cell in NOTEBOOK_LAYOUT['cells']:
            marker = '# TrackID3x3 Kaggle cell: ' + cell['id']
            if cell['cell_type'] == 'code' and raw.startswith(marker + '\n'):
                observed[cell['id']] = raw.strip() + '\n'
    rebuilt = json.loads(json.dumps(NOTEBOOK_LAYOUT))
    for cell in rebuilt['cells']:
        if cell['cell_type'] == 'code':
            if cell['id'] not in observed:
                raise RuntimeError(f'No executed source for {cell["id"]}; use Restart Session → Run All.')
            cell['source'] = observed[cell['id']].splitlines(keepends=True)
    path = RUN_DIR / 'kaggle_source_snapshot.ipynb'
    path.write_text(json.dumps(rebuilt, ensure_ascii=False, indent=1) + '\n')
    save_json('source_hashes.json', dict(original_notebook_sha256=ORIGINAL_NOTEBOOK_SHA256,
              kaggle_notebook_sha256=file_sha256(path), kaggle_notebook_path=str(path),
              hash_basis='Executed raw code + build markdown/metadata; output-free source notebook. Markdown-only edits in the UI are not captured.'))


In [ ]:
# TrackID3x3 Kaggle cell: environment
with baseline_stage(1, 'Environment'):
    install_missing_dependencies()
    # ==============================
    # 1. Environment preparation
    # ==============================


    import os
    import glob
    import re
    import cv2
    import ndjson
    import math
    import colorsys
    import torch
    import easyocr

    import numpy as np
    import pandas as pd

    from collections import defaultdict
    from scipy.optimize import linear_sum_assignment
    from ultralytics import YOLO
    from IPython.display import HTML
    from base64 import b64encode


    save_environment()
    print('Run directory:', RUN_DIR)


In [ ]:
# TrackID3x3 Kaggle cell: data
with baseline_stage(2, 'Data'):
    video_path, gt_file, keypoint_file, court_image_path = resolve_data()
    YOLO_WEIGHTS = prepare_models()


In [ ]:
# TrackID3x3 Kaggle cell: label-probe
with baseline_stage('3-probe', 'YOLO / ByteTrack real-label compatibility'):
    probe_source = '"""Isolated GPU tracking probe. Embedded in the notebook and run in a subprocess.\n\nNo probe tracker state or observations enter the production baseline.\n"""\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport sys\nimport traceback\nfrom ultralytics import YOLO\n\ndef get_frame_id_from_filename(path):\n    """\n    Extract the trailing frame number from a label filename.\n\n    Example:\n        000123.txt -> 123\n        IMG_0104_1_123.txt -> 123\n    """\n\n    base = os.path.splitext(os.path.basename(path))[0]\n\n    match = re.search(r"(\\d+)$", base)\n\n    if not match:\n        raise ValueError(\n            f"No frame number found in filename: {base}"\n        )\n\n    return int(match.group(1))\n\ndef parse_yolo_tracking_line(parts):\n    """\n    Parse one YOLO tracking-label line.\n\n    Returns:\n        track_id, confidence, xc, yc, width, height\n    """\n\n    xc, yc, w_rel, h_rel = map(float, parts[1:5])\n\n    if len(parts) >= 7:\n        a, b = parts[5], parts[6]\n\n        try:\n            track_id = int(a)\n            confidence = float(b)\n\n        except ValueError:\n            confidence = float(a)\n            track_id = int(b)\n\n    elif len(parts) == 6:\n\n        track_id = int(parts[5])\n        confidence = 1.0\n\n    else:\n        raise ValueError(\n            f"Cannot parse YOLO tracking line: {parts}"\n        )\n\n    return (\n        track_id,\n        confidence,\n        xc,\n        yc,\n        w_rel,\n        h_rel\n    )\n\n\n\ndef check_result_label(parts, box):\n    """Compare the original parser with the actual tracked Result object."""\n    parsed = parse_yolo_tracking_line(parts)\n    expected = (int(box.id.item()), float(box.conf.item()), *box.xywhn.reshape(-1).tolist())\n    if len(parts) != 7 or parsed[0] != expected[0] or not all(\n        math.isclose(a, b, rel_tol=1e-4, abs_tol=1e-6) for a, b in zip(parsed[1:], expected[1:])\n    ):\n        raise RuntimeError(f\'Original parser does not match actual tracked labels: line={parts}, parsed={parsed}, expected={expected}. Inspect evidence; no speculative parser patch was applied.\')\n    return {\'line\': \' \'.join(parts), \'parsed\': list(parsed), \'expected_from_result\': list(expected)}\n\n\ndef main():\n    video, weights, run_dir, max_frames = sys.argv[1:]\n    run_dir = Path(run_dir)\n    destination = run_dir / \'label_probe\'\n    destination.mkdir(exist_ok=False)\n    summary = dict(status=\'RUNNING\', frames=0, tracked_label_rows=0, examples=[], parser_changed=False)\n    try:\n        model = YOLO(weights)\n        results = model.track(source=video, tracker=str(run_dir / \'bytetrack.yaml\'), device=\'0\',\n                              classes=[0], save=False, save_txt=True, save_conf=True,\n                              stream=True, verbose=False, project=str(destination), name=\'tracking\', exist_ok=False)\n        previous = set()\n        try:\n            for index, result in enumerate(results, start=1):\n                summary[\'frames\'] = index\n                labels = set((Path(result.save_dir) / \'labels\').glob(\'*.txt\'))\n                new_labels = labels - previous\n                previous = labels\n                boxes = result.boxes\n                if boxes is not None and len(boxes):\n                    if not boxes.is_track:\n                        raise RuntimeError(\'Actual output contains untracked detection-only boxes. The original six-field parser has not been verified for this format.\')\n                    if len(new_labels) != 1:\n                        raise RuntimeError(f\'Expected one actual saved label file at sample frame {index}; found {new_labels}\')\n                    label = next(iter(new_labels))\n                    lines = label.read_text().splitlines()\n                    if len(lines) != len(boxes):\n                        raise RuntimeError(f\'Saved label count differs from Results boxes in {label}\')\n                    for line, box in zip(lines, boxes):\n                        item = check_result_label(line.split(), box)\n                        summary[\'tracked_label_rows\'] += 1\n                        if len(summary[\'examples\']) < 5:\n                            summary[\'examples\'].append(dict(frame=index, file=str(label), **item))\n                if index >= int(max_frames):\n                    break\n        finally:\n            results.close()\n        if not summary[\'tracked_label_rows\']:\n            raise RuntimeError(\'No tracked label rows in real sample; parser compatibility is unverified. Inspect the original video and runtime.\')\n        summary[\'status\'] = \'PASS\'\n    except BaseException:\n        summary[\'status\'] = \'FAIL\'\n        summary[\'error\'] = traceback.format_exc()\n        raise\n    finally:\n        (destination / \'label_format.json\').write_text(json.dumps(summary, indent=2) + \'\\n\')\n    print(json.dumps(summary, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n'
    probe_path = TEMP_DIR / 'label_probe.py'
    probe_path.write_text(probe_source)
    process = subprocess.run([sys.executable, str(probe_path), video_path, YOLO_WEIGHTS, str(RUN_DIR), str(PROBE_FRAMES)],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    (RUN_DIR / 'label_probe.log').write_text(process.stdout)
    if process.returncode:
        raise RuntimeError('Real tracking/parser compatibility probe failed. Inspect label_probe.log and label_probe/label_format.json; no parser behavior was changed.')
    probe_summary = json.loads((RUN_DIR / 'label_probe' / 'label_format.json').read_text())
    print('Actual sample:', probe_summary['frames'], 'frames;', probe_summary['tracked_label_rows'], 'parsed tracked rows')


In [ ]:
# TrackID3x3 Kaggle cell: tracking
with baseline_stage(3, 'Detection + Tracking'):
    # ==============================
    # 3. Player detection and tracking
    # ==============================

    # Load YOLO11x
    model = YOLO(YOLO_WEIGHTS)

    # Run ByteTrack in streaming mode.
    # stream=True prevents Results objects for all video frames
    # from accumulating in RAM.
    results = model.track(
        source=video_path,
        tracker=str(RUN_DIR / "bytetrack.yaml"),
        device="0",
        classes=[0],       # COCO class 0: person
        save=SAVE_VIDEOS,
        save_txt=True,
        save_conf=True,
        stream=True,
        verbose=False,
        project=str(RUN_DIR),
        name="tracking",
        exist_ok=False
    )

    # Consume the generator frame by frame.
    # Only the current Result object is kept in memory.
    out_dir = None

    for result in results:
        if out_dir is None:
            out_dir = str(result.save_dir)

    if out_dir is None:
        raise RuntimeError(
            "Tracking produced no results."
        )

    print("Output directory:", out_dir)
    if Path(out_dir).resolve() != (RUN_DIR / 'tracking').resolve():
        raise RuntimeError(f'Unexpected tracking save directory: {out_dir}')
    print('Saved label files:', len(list((Path(out_dir) / 'labels').glob('*.txt'))))
    save_json('tracking_arguments.json', vars(model.predictor.args))


In [ ]:
# TrackID3x3 Kaggle cell: mot
with baseline_stage(4, 'MOT Conversion'):
    # ==============================================================
    # 4. Convert YOLO tracking output to MOT format
    # ==============================================================


    def get_frame_id_from_filename(path):
        """
        Extract the trailing frame number from a label filename.

        Example:
            000123.txt -> 123
            IMG_0104_1_123.txt -> 123
        """

        base = os.path.splitext(os.path.basename(path))[0]

        match = re.search(r"(\d+)$", base)

        if not match:
            raise ValueError(
                f"No frame number found in filename: {base}"
            )

        return int(match.group(1))


    def parse_yolo_tracking_line(parts):
        """
        Parse one YOLO tracking-label line.

        Returns:
            track_id, confidence, xc, yc, width, height
        """

        xc, yc, w_rel, h_rel = map(float, parts[1:5])

        if len(parts) >= 7:
            a, b = parts[5], parts[6]

            try:
                track_id = int(a)
                confidence = float(b)

            except ValueError:
                confidence = float(a)
                track_id = int(b)

        elif len(parts) == 6:

            track_id = int(parts[5])
            confidence = 1.0

        else:
            raise ValueError(
                f"Cannot parse YOLO tracking line: {parts}"
            )

        return (
            track_id,
            confidence,
            xc,
            yc,
            w_rel,
            h_rel
        )


    # --------------------------------------------------
    # Create MOT-format file
    # --------------------------------------------------

    mot_txt_path = os.path.join(
        out_dir,
        "mot_results.txt"
    )

    label_dir = os.path.join(
        out_dir,
        "labels"
    )

    label_files = sorted(
        glob.glob(
            os.path.join(label_dir, "*.txt")
        ),
        key=get_frame_id_from_filename
    )

    # Read video properties
    cap = cv2.VideoCapture(video_path)

    width = int(
        cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    )

    height = int(
        cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )

    fps = cap.get(
        cv2.CAP_PROP_FPS
    )

    cap.release()


    with open(mot_txt_path, "w") as mot_file:

        for label_file in label_files:

            frame_id = get_frame_id_from_filename(
                label_file
            )

            with open(label_file, "r") as f:

                for line in f:

                    parts = line.strip().split()

                    if len(parts) < 6:
                        continue

                    (
                        track_id,
                        confidence,
                        xc,
                        yc,
                        w_rel,
                        h_rel
                    ) = parse_yolo_tracking_line(parts)

                    # Relative coordinates -> pixel coordinates
                    w_pix = w_rel * width
                    h_pix = h_rel * height

                    x_pix = (
                        xc * width
                        - w_pix / 2
                    )

                    y_pix = (
                        yc * height
                        - h_pix / 2
                    )

                    # MOTChallenge-style format
                    mot_file.write(
                        f"{frame_id},"
                        f"{track_id},"
                        f"{x_pix:.2f},"
                        f"{y_pix:.2f},"
                        f"{w_pix:.2f},"
                        f"{h_pix:.2f},"
                        f"{confidence:.2f},"
                        "-1,-1,-1\n"
                    )

    print("MOT file:", mot_txt_path)


    # --------------------------------------------------

    with open(mot_txt_path) as handle:
        mot_rows = [line for line in handle if line.strip()]
    if not mot_rows:
        raise RuntimeError('MOT conversion produced no rows; inspect tracking labels.')
    print('MOT rows:', len(mot_rows))


In [ ]:
# TrackID3x3 Kaggle cell: homography
with baseline_stage(5, 'Homography'):
    # ==============================
    # 5. Outdoor court homography
    # ==============================

    with open(keypoint_file, "r") as f:
        data = ndjson.load(f)

    # Retrieve the Labelbox project
    project = next(
        iter(
            data[0]["projects"].values()
        )
    )

    annotations = (
        project["labels"][0]
        ["annotations"]["objects"]
    )

    # Outdoor court coordinates in centimeters
    keypoint_mapping = {
        "rku_0":  [0, 0],
        "rku_1":  [0, 1105],
        "rku_10": [1505, 0],
        "rku_11": [1505, 1105],
    }

    image_points = []
    court_points = []

    for obj in annotations:

        name = obj["value"]

        if name not in keypoint_mapping:
            continue

        x = obj["point"]["x"]
        y = obj["point"]["y"]

        image_points.append(
            [x, y]
        )

        court_points.append(
            keypoint_mapping[name]
        )


    image_points = np.asarray(
        image_points,
        dtype=np.float32
    )

    court_points = np.asarray(
        court_points,
        dtype=np.float32
    )

    if len(image_points) != 4:

        raise ValueError(
            "Expected 4 Outdoor court keypoints, "
            f"but found {len(image_points)}."
        )


    print("Image coordinates:")
    print(image_points)

    print("\nCourt coordinates [cm]:")
    print(court_points)


    # Compute homography
    H, _ = cv2.findHomography(
        image_points,
        court_points
    )

    if H is None:

        raise RuntimeError(
            "Failed to calculate the homography."
        )

    print("\nHomography:")
    print(H)


    def homographic_transformation(x, y, H):
        """
        Transform one image point into Outdoor court
        coordinates in centimeters.
        """

        point = np.array(
            [x, y, 1.0],
            dtype=np.float64
        ).reshape(3, 1)

        transformed = H @ point

        transformed /= transformed[
            2, 0
        ]

        return (
            float(transformed[0, 0]),
            float(transformed[1, 0])
        )


    # Outdoor court boundaries [cm]
    min_x = 0.0
    max_x = 1505.0

    min_y = 0.0
    max_y = 1105.0

    print(
        "\nCourt range [cm]: "
        f"x=[{min_x}, {max_x}], "
        f"y=[{min_y}, {max_y}]"
    )
    save_json('homography.json', dict(H=H.tolist(), image_points_px=image_points.tolist(), court_points_cm=court_points.tolist()))
    np.save(RUN_DIR / 'homography.npy', H)


In [ ]:
# TrackID3x3 Kaggle cell: court-filtering
with baseline_stage(6, 'Court Filtering'):
    # ==========================================
    # 6. Outdoor court filtering
    # ==========================================

    columns = [
        "frame_id",
        "id",
        "x",
        "y",
        "width",
        "height",
        "conf",
        "class",
        "visibility",
        "empty",
    ]

    df = pd.read_csv(
        mot_txt_path,
        names=columns,
        sep=","
    )


    # --------------------------------------------------
    # Convert bbox bottom centers to court coordinates
    # --------------------------------------------------

    df["bot_cx"] = (
        df["x"]
        + df["width"] / 2
    )

    df["bot_cy"] = (
        df["y"]
        + df["height"]
    )

    df[
        ["court_x", "court_y"]
    ] = df.apply(
        lambda row:
            homographic_transformation(
                row["bot_cx"],
                row["bot_cy"],
                H
            ),
        axis=1,
        result_type="expand"
    )


    # --------------------------------------------------
    # Track-level Outdoor filtering
    # --------------------------------------------------

    paint_line_y = 1002.5
    paint_x_min = 502.5
    paint_x_max = 1002.5

    valid_ids = []

    for track_id, group in df.groupby(
        "id"
    ):

        total = group[
            "frame_id"
        ].nunique()

        inside = group[
            (group["court_x"] >= min_x)
            &
            (group["court_x"] <= max_x)
            &
            (group["court_y"] >= min_y)
            &
            (group["court_y"] <= max_y)
        ]

        # Longest consecutive run inside the court
        frames_inside = sorted(
            inside[
                "frame_id"
            ].unique()
        )

        max_run = 0
        current_run = 0
        previous_frame = None

        for frame_id in frames_inside:

            if (
                previous_frame is not None
                and
                frame_id
                == previous_frame + 1
            ):
                current_run += 1

            else:
                current_run = 1

            max_run = max(
                max_run,
                current_run
            )

            previous_frame = frame_id


        # Near-side endline region
        c1 = (
            group["court_y"].between(
                -300,
                100
            )
            &
            group["court_x"].between(
                0,
                1505
            )
        ).sum()


        # Far-side region outside the paint extension
        c2 = (
            (group["court_y"] > paint_line_y)
            &
            (
                (group["court_x"] < paint_x_min)
                |
                (group["court_x"] > paint_x_max)
            )
        ).sum()


        if (
            max_run >= 10
            and
            c1 <= total / 2
            and
            c2 <= total / 2
        ):
            valid_ids.append(
                track_id
            )


    df = df[
        df["id"].isin(
            valid_ids
        )
    ].copy()


    # --------------------------------------------------
    # Detection-level filtering
    # --------------------------------------------------

    buffer_cm = 300

    df = df[
        (df["court_x"] >= min_x - buffer_cm)
        &
        (df["court_x"] <= max_x + buffer_cm)
        &
        (df["court_y"] >= min_y - buffer_cm)
        &
        (df["court_y"] <= max_y + buffer_cm)
    ].copy()


    df["frame_id"] = (
        df["frame_id"].astype(int)
    )

    df["id"] = (
        df["id"].astype(int)
    )


    # --------------------------------------------------
    # Read the actual video length
    # --------------------------------------------------

    cap = cv2.VideoCapture(
        video_path
    )

    total_video_frames = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    cap.release()


    # --------------------------------------------------
    # Save filtered tracklets
    # --------------------------------------------------

    out_bbox = os.path.join(
        out_dir,
        "filtered_mot.txt"
    )

    df[
        [
            "frame_id",
            "id",
            "x",
            "y",
            "width",
            "height",
            "conf",
        ]
    ].to_csv(
        out_bbox,
        index=False,
        header=False
    )


    out_court = os.path.join(
        out_dir,
        "transformed_court.txt"
    )

    df[
        [
            "frame_id",
            "id",
            "court_x",
            "court_y",
        ]
    ].to_csv(
        out_court,
        index=False,
        header=False
    )


    print("Outdoor filtering completed.")

    print(
        "Remaining ByteTrack IDs:",
        sorted(
            df["id"].unique()
        )
    )

    print(
        "Number of tracklets:",
        df["id"].nunique()
    )

    print(
        "Number of video frames:",
        total_video_frames
    )

    print(
        "Filtered MOT file:",
        out_bbox
    )

    print(
        "Court-coordinate file:",
        out_court
    )
    print('Filtered rows:', len(df))


In [ ]:
# TrackID3x3 Kaggle cell: ocr
with baseline_stage(7, 'OCR'):
    # ==========================================================
    # 7. Team identification and jersey-number recognition
    # ==========================================================

    # --------------------------------------------------
    # Fixed roster for this tutorial clip
    # --------------------------------------------------

    team_rosters = {
        "A": {4, 10, 12},
        "B": {6, 9, 11},
    }

    all_valid_jerseys = set().union(
        *team_rosters.values()
    )

    identity_order = [
        ("A", 4),
        ("A", 10),
        ("A", 12),
        ("B", 6),
        ("B", 9),
        ("B", 11),
    ]


    # --------------------------------------------------
    # OCR and team-color settings
    # --------------------------------------------------

    reader = easyocr.Reader(
        ["en"],
        gpu=True,
        model_storage_directory=str(OCR_MODEL_DIR),
        download_enabled=ALLOW_DOWNLOADS
    )


    if str(reader.device) != 'cuda':
        raise RuntimeError(f'EasyOCR did not select CUDA: {reader.device}. CPU fallback is disabled.')
    print('EasyOCR device:', reader.device)

    MAX_SAMPLED_FRAMES_PER_TRACK = 40
    MIN_OCR_CONFIDENCE = 0.10

    # Histogram settings for team identification
    H_BINS = 16
    S_BINS = 8


    # --------------------------------------------------
    # Helper functions
    # --------------------------------------------------

    def get_upper_body_crop(frame, x, y, w, h):
        """
        Crop the upper-body region from a player bounding box.
        """

        x1 = int(max(0, x))
        y1 = int(max(0, y))

        x2 = int(min(frame.shape[1], x + w))
        y2 = int(min(frame.shape[0], y + h))

        crop = frame[y1:y2, x1:x2]

        if crop.size == 0:
            return None

        H = crop.shape[0]
        W = crop.shape[1]

        # Center-upper torso region
        tx1 = int(0.20 * W)
        tx2 = int(0.80 * W)
        ty1 = int(0.12 * H)
        ty2 = int(0.62 * H)

        torso = crop[ty1:ty2, tx1:tx2]

        if torso.size == 0:
            return None

        return torso


    def extract_color_feature(image):
        """
        Extract an HSV histogram feature from a torso crop.
        """

        hsv = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2HSV
        )

        hist = cv2.calcHist(
            [hsv],
            [0, 1],
            None,
            [H_BINS, S_BINS],
            [0, 180, 0, 256]
        )

        hist = cv2.normalize(
            hist,
            None
        ).flatten()

        return hist.astype(np.float32)


    def infer_roster_team_from_jersey(jersey_number):
        """
        Infer the team from the roster if the jersey number
        belongs to exactly one team.
        """

        candidate_teams = [
            team_id
            for team_id, roster in team_rosters.items()
            if jersey_number in roster
        ]

        if len(candidate_teams) == 1:
            return candidate_teams[0]

        return None


    # --------------------------------------------------
    # Sample frames from each ByteTrack tracklet
    # --------------------------------------------------

    sampled_groups = []

    for track_id, group in df.groupby(
        "id"
    ):
        group = group.sort_values(
            "frame_id"
        )

        n_samples = min(
            MAX_SAMPLED_FRAMES_PER_TRACK,
            len(group)
        )

        sample_indices = np.linspace(
            0,
            len(group) - 1,
            n_samples,
            dtype=int
        )

        sample_indices = np.unique(
            sample_indices
        )

        sampled_groups.append(
            group.iloc[sample_indices]
        )

    if not sampled_groups:
        raise RuntimeError(
            "No tracklets remain after court filtering."
        )

    sample_df = pd.concat(
        sampled_groups,
        ignore_index=True
    )

    samples_by_frame = {
        int(frame_id): group.to_dict("records")
        for frame_id, group
        in sample_df.groupby("frame_id")
    }


    # --------------------------------------------------
    # Run OCR and color-feature extraction
    # --------------------------------------------------

    ocr_scores = defaultdict(
        lambda: defaultdict(float)
    )

    track_color_features = defaultdict(list)

    cap = cv2.VideoCapture(
        video_path
    )

    frame_id = 0

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame_id += 1

        if frame_id not in samples_by_frame:
            continue

        for row in samples_by_frame[frame_id]:

            track_id = int(
                row["id"]
            )

            x = row["x"]
            y = row["y"]
            w = row["width"]
            h = row["height"]

            # --------------------------
            # OCR on full player bbox
            # --------------------------

            x1 = int(max(0, x))
            y1 = int(max(0, y))
            x2 = int(min(frame.shape[1], x + w))
            y2 = int(min(frame.shape[0], y + h))

            player_crop = frame[
                y1:y2,
                x1:x2
            ]

            if player_crop.size != 0:

                ocr_crop = cv2.resize(
                    player_crop,
                    None,
                    fx=2.0,
                    fy=2.0,
                    interpolation=cv2.INTER_CUBIC
                )

                results = reader.readtext(
                    ocr_crop,
                    allowlist="0123456789",
                    detail=1,
                    paragraph=False
                )

                for _, text, confidence in results:

                    if confidence < MIN_OCR_CONFIDENCE:
                        continue

                    digits = re.sub(
                        r"\D",
                        "",
                        text
                    )

                    if digits == "":
                        continue

                    jersey_number = int(digits)

                    if jersey_number not in all_valid_jerseys:
                        continue

                    ocr_scores[
                        track_id
                    ][
                        jersey_number
                    ] += float(confidence)

            # --------------------------
            # Team feature on upper body
            # --------------------------

            torso = get_upper_body_crop(
                frame,
                x,
                y,
                w,
                h
            )

            if torso is not None:

                feature = extract_color_feature(
                    torso
                )

                track_color_features[
                    track_id
                ].append(feature)

    cap.release()


    # --------------------------------------------------
    # Assign a jersey number to each tracklet
    # --------------------------------------------------

    track_to_jersey = {}

    print("Jersey OCR results:")

    for track_id in sorted(df["id"].unique()):

        scores = ocr_scores.get(
            track_id,
            {}
        )

        if not scores:
            track_to_jersey[
                track_id
            ] = None

            print(
                f"Track {track_id}: "
                "no valid jersey number recognized"
            )

            continue

        jersey_number = max(
            scores,
            key=scores.get
        )

        track_to_jersey[
            track_id
        ] = jersey_number

        score_string = ", ".join(
            f"{number}:{score:.2f}"
            for number, score
            in sorted(
                scores.items(),
                key=lambda item: -item[1]
            )
        )

        print(
            f"Track {track_id} -> "
            f"jersey {jersey_number} "
            f"({score_string})"
        )


    # --------------------------------------------------


In [ ]:
# TrackID3x3 Kaggle cell: teams
with baseline_stage(8, 'Team Assignment'):
    # Team identification by jersey-color clustering
    # --------------------------------------------------

    track_feature_vectors = {}

    for track_id, features in track_color_features.items():

        if len(features) == 0:
            continue

        track_feature_vectors[
            track_id
        ] = np.mean(
            np.stack(features, axis=0),
            axis=0
        ).astype(np.float32)

    if len(track_feature_vectors) < 2:
        raise RuntimeError(
            "Not enough tracklets with color features to run "
            "team identification."
        )

    cluster_track_ids = sorted(
        track_feature_vectors.keys()
    )

    feature_matrix = np.stack(
        [
            track_feature_vectors[track_id]
            for track_id in cluster_track_ids
        ],
        axis=0
    ).astype(np.float32)

    criteria = (
        cv2.TERM_CRITERIA_EPS
        + cv2.TERM_CRITERIA_MAX_ITER,
        100,
        1.0
    )

    compactness, labels, centers = cv2.kmeans(
        feature_matrix,
        2,
        None,
        criteria,
        20,
        cv2.KMEANS_PP_CENTERS
    )

    track_to_cluster = {
        track_id: int(label[0])
        for track_id, label
        in zip(cluster_track_ids, labels)
    }

    print("\nTeam-cluster assignment:")

    for track_id in sorted(track_to_cluster):

        print(
            f"Track {track_id} -> cluster {track_to_cluster[track_id]}"
        )


    # --------------------------------------------------
    # Map clusters to Team A / Team B
    # using the initial offense position
    # --------------------------------------------------

    # In the original TrackID3x3 Outdoor baseline,
    # for a check-ball segment starting at the top,
    # the player closest to the midpoint of the outer court line
    # parallel to the end line is identified as the initial offense.
    #
    # In this tutorial:
    #   Team A = initial offense
    #   Team B = initial defense
    #
    # Team separation itself has already been performed by
    # jersey-color clustering. Here, the initial position is used
    # only to resolve which of the two arbitrary cluster labels
    # corresponds to Team A.

    eligible_track_ids = set(
        track_to_cluster.keys()
    )

    eligible_df = df[
        df["id"].isin(eligible_track_ids)
    ].copy()


    # --------------------------------------------------
    # Use the first available frame
    # --------------------------------------------------

    initial_frame = int(
        eligible_df["frame_id"].min()
    )

    initial_df = eligible_df[
        eligible_df["frame_id"] == initial_frame
    ].copy()

    if len(initial_df) == 0:
        raise RuntimeError(
            "No player detections were available in the initial frame."
        )


    # --------------------------------------------------
    # Find the initial offensive player
    # --------------------------------------------------

    # Midpoint of the outer court line parallel to the end line
    # [cm]

    top_reference = np.array(
        [
            (min_x + max_x) / 2.0,
            max_y
        ],
        dtype=np.float32
    )

    initial_positions = initial_df[
        [
            "court_x",
            "court_y"
        ]
    ].to_numpy(
        dtype=np.float32
    )

    distances_to_top = np.linalg.norm(
        initial_positions - top_reference,
        axis=1
    )

    offense_row = initial_df.iloc[
        int(np.argmin(distances_to_top))
    ]

    offense_track_id = int(
        offense_row["id"]
    )

    offense_cluster = track_to_cluster[
        offense_track_id
    ]


    # --------------------------------------------------
    # Find the initial defensive player
    # --------------------------------------------------

    # Following the original Outdoor baseline,
    # the player closest to the identified offensive player
    # is treated as the initial defender.

    other_players = initial_df[
        initial_df["id"] != offense_track_id
    ].copy()

    if len(other_players) == 0:
        raise RuntimeError(
            "No candidate defender was available in the initial frame."
        )

    offense_position = np.array(
        [
            offense_row["court_x"],
            offense_row["court_y"]
        ],
        dtype=np.float32
    )

    other_positions = other_players[
        [
            "court_x",
            "court_y"
        ]
    ].to_numpy(
        dtype=np.float32
    )

    distances_to_offense = np.linalg.norm(
        other_positions - offense_position,
        axis=1
    )

    defense_row = other_players.iloc[
        int(np.argmin(distances_to_offense))
    ]

    defense_track_id = int(
        defense_row["id"]
    )

    defense_cluster = track_to_cluster[
        defense_track_id
    ]


    # --------------------------------------------------
    # Resolve the arbitrary k-means cluster labels
    # --------------------------------------------------

    clusters = sorted(
        set(track_to_cluster.values())
    )

    if len(clusters) != 2:
        raise RuntimeError(
            f"Expected 2 team clusters, but found {len(clusters)}."
        )

    if offense_cluster == defense_cluster:
        raise RuntimeError(
            "The initial offense and defense were assigned to the "
            "same color cluster. Team-cluster alignment is ambiguous."
        )

    cluster_to_team = {
        offense_cluster: "A",
        defense_cluster: "B",
    }

    print(
        "\nInitial team anchors:"
    )

    print(
        f"Frame {initial_frame}: "
        f"Track {offense_track_id} -> initial offense -> Team A"
    )

    print(
        f"Frame {initial_frame}: "
        f"Track {defense_track_id} -> initial defense -> Team B"
    )

    print(
        "\nCluster-to-team mapping:"
    )

    for cluster_id in sorted(cluster_to_team):

        print(
            f"Cluster {cluster_id} "
            f"-> Team {cluster_to_team[cluster_id]}"
        )


    # --------------------------------------------------


In [ ]:
# TrackID3x3 Kaggle cell: identity
with baseline_stage(9, 'Semantic Identity'):
    # Build semantic identities
    # --------------------------------------------------

    identity_df = df.copy()

    identity_df["jersey_number"] = identity_df["id"].map(
        track_to_jersey
    )

    identity_df["cluster_id"] = identity_df["id"].map(
        track_to_cluster
    )

    identity_df["team_id"] = identity_df["cluster_id"].map(
        cluster_to_team
    )

    # Remove rows without either a team or a jersey
    identity_df = identity_df[
        identity_df["jersey_number"].notna()
        &
        identity_df["team_id"].notna()
    ].copy()

    identity_df["jersey_number"] = identity_df[
        "jersey_number"
    ].astype(int)

    # Keep only roster-compatible identities
    identity_df = identity_df[
        identity_df.apply(
            lambda row:
                row["jersey_number"]
                in team_rosters[row["team_id"]],
            axis=1
        )
    ].copy()

    identity_df["identity"] = (
        identity_df["team_id"]
        +
        identity_df["jersey_number"].astype(str)
    )

    print("\nTracklet identities:")

    for track_id in sorted(identity_df["id"].unique()):

        tmp = identity_df[
            identity_df["id"] == track_id
        ]

        identity = tmp["identity"].iloc[0]

        print(
            f"Track {track_id} -> {identity}"
        )


    # --------------------------------------------------
    # Merge fragmented tracklets through semantic identity
    # --------------------------------------------------

    identity_df = (
        identity_df
        .sort_values(
            "conf",
            ascending=False
        )
        .drop_duplicates(
            subset=["frame_id", "identity"],
            keep="first"
        )
    )

    recognized_identities = set(
        identity_df["identity"].unique()
    )

    required_identities = {
        team_id + str(jersey_number)
        for team_id, jersey_number
        in identity_order
    }

    missing_identities = (
        required_identities
        - recognized_identities
    )

    if missing_identities:
        raise RuntimeError(
            "The following player identities were not recognized: "
            + ", ".join(sorted(missing_identities))
        )


    # --------------------------------------------------


In [ ]:
# TrackID3x3 Kaggle cell: prediction
with baseline_stage(10, 'Final Prediction'):
    # Build the final prediction table
    # --------------------------------------------------

    # The final prediction is the single source of truth for:
    #   1. TI-HOTA evaluation
    #   2. prediction minimap
    #   3. video overlay
    #
    # Challenge submission fields:
    #   frame_id, court_x, court_y, team_id, jersey_number
    #
    # video_id and bbox columns are retained internally only
    # for bookkeeping and visualization.

    prediction_tables = []

    video_id = os.path.splitext(
        os.path.basename(video_path)
    )[0]

    # Court position is the evaluated prediction.
    # Bbox width/height are auxiliary visualization quantities.
    # Bbox x/y will be reconstructed from the final court position
    # through the inverse homography, so the video overlay and
    # minimap refer to exactly the same predicted position.

    interpolation_columns = [
        "court_x",
        "court_y",
        "width",
        "height",
    ]

    H_inv = np.linalg.inv(H)

    for team_id, jersey_number in identity_order:

        identity = (
            team_id
            + str(jersey_number)
        )

        player = identity_df[
            identity_df["identity"] == identity
        ].copy()

        player = (
            player
            .sort_values("frame_id")
            .set_index("frame_id")
        )

        # Only predict between this player's first and last
        # observed frames. Do not extend the track to the
        # beginning or end of the entire video.
        full_frame_index = range(
            int(player.index.min()),
            int(player.index.max()) + 1
        )

        player = player.reindex(
            full_frame_index
        )

        # Interpolate gaps between observations.
        player[interpolation_columns] = (
            player[interpolation_columns]
            .interpolate(
                method="linear",
                limit_direction="both"
            )
        )

        player = (
            player
            .reset_index()
            .rename(
                columns={"index": "frame_id"}
            )
        )

        # Map the final evaluated court positions back to
        # image-space foot points. Width/height are retained
        # from the tracking output (with interpolation),
        # because inverse homography alone cannot recover bbox size.

        court_points_final = player[
            ["court_x", "court_y"]
        ].to_numpy(
            dtype=np.float32
        ).reshape(
            -1,
            1,
            2
        )

        image_foot_points = cv2.perspectiveTransform(
            court_points_final,
            H_inv.astype(np.float32)
        ).reshape(
            -1,
            2
        )

        bbox_width = player[
            "width"
        ].to_numpy(
            dtype=float
        )

        bbox_height = player[
            "height"
        ].to_numpy(
            dtype=float
        )

        bbox_x = (
            image_foot_points[:, 0]
            - bbox_width / 2.0
        )

        bbox_y = (
            image_foot_points[:, 1]
            - bbox_height
        )

        output = pd.DataFrame({
            "video_id": video_id,
            "frame_id": player["frame_id"].astype(int),
            "court_x": player["court_x"],
            "court_y": player["court_y"],
            "team_id": team_id,
            "jersey_number": jersey_number,
            "bbox_x": bbox_x,
            "bbox_y": bbox_y,
            "bbox_width": bbox_width,
            "bbox_height": bbox_height,
        })

        prediction_tables.append(output)

    predictions_df = pd.concat(
        prediction_tables,
        ignore_index=True
    )

    predictions_df = predictions_df.sort_values(
        [
            "frame_id",
            "team_id",
            "jersey_number"
        ]
    ).reset_index(
        drop=True
    )


    # --------------------------------------------------
    # Save the final prediction
    # --------------------------------------------------

    predictions_file = os.path.join(
        out_dir,
        "predictions.csv"
    )

    # Challenge submission format
    submission_columns = [
        "frame_id",
        "court_x",
        "court_y",
        "team_id",
        "jersey_number",
    ]

    predictions_df[
        submission_columns
    ].to_csv(
        predictions_file,
        index=False,
        float_format="%.5f"
    )

    print("\nFinal player identities:")

    print(
        predictions_df[
            ["team_id", "jersey_number"]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )

    print(
        "\nPrediction file:",
        predictions_file
    )

    print(
        "Rows:",
        len(predictions_df)
    )

    print(
        "Frames:",
        predictions_df["frame_id"].nunique()
    )


In [ ]:
# TrackID3x3 Kaggle cell: evaluator-definitions
# ==============================
# 8. TI-HOTA evaluation functions
# ==============================


def loc_sim(pred_position, gt_position, tau):
    """
    Compute location similarity.

    Coordinates and tau must use the same unit.
    In this tutorial they are expressed in centimeters.
    """

    pred_position = np.asarray(
        pred_position,
        dtype=float
    )

    gt_position = np.asarray(
        gt_position,
        dtype=float
    )

    distance_squared = np.sum(
        (
            pred_position
            - gt_position
        ) ** 2
    )

    return math.exp(
        math.log(0.05)
        *
        distance_squared
        /
        (tau ** 2)
    )


def id_sim(pred_attribute, gt_attribute):
    """
    Identity similarity is 1 only when both
    team ID and jersey number are correct.
    """

    return int(
        pred_attribute
        == gt_attribute
    )


def sim_ti_hota(prediction, ground_truth, tau):
    """Combined TI-HOTA similarity."""

    return (
        loc_sim(
            prediction["position"],
            ground_truth["position"],
            tau
        )
        *
        id_sim(
            prediction["attributes"],
            ground_truth["attributes"]
        )
    )


def compute_ti_hota_alpha(
    frames,
    tau,
    alpha
):

    total_tp = 0
    total_fp = 0
    total_fn = 0

    association_counts = {}
    false_positive_association_counts = {}


    for frame in frames:

        gt_objects = frame.get(
            "ground_truths",
            []
        )

        pred_objects = frame.get(
            "predictions",
            []
        )

        n_gt = len(
            gt_objects
        )

        n_pred = len(
            pred_objects
        )


        # Similarity matrix
        combined = np.zeros(
            (
                n_gt,
                n_pred
            )
        )

        for i, gt in enumerate(
            gt_objects
        ):

            for j, pred in enumerate(
                pred_objects
            ):

                combined[
                    i, j
                ] = (
                    loc_sim(
                        pred["position"],
                        gt["position"],
                        tau
                    )
                    *
                    id_sim(
                        pred["attributes"],
                        gt["attributes"]
                    )
                )


        current_gt_matches = {}


        if (
            n_gt > 0
            and
            n_pred > 0
        ):

            cost = (
                1.0
                - combined
            )

            cost[
                combined < alpha
            ] = 1e6

            row_indices, col_indices = (
                linear_sum_assignment(
                    cost
                )
            )

            for i, j in zip(
                row_indices,
                col_indices
            ):

                if (
                    combined[i, j]
                    >= alpha
                ):
                    current_gt_matches[
                        i
                    ] = j


        # Detection counts
        tp = len(
            current_gt_matches
        )

        fp = (
            n_pred
            - tp
        )

        fn = (
            n_gt
            - tp
        )

        total_tp += tp
        total_fp += fp
        total_fn += fn


        # --------------------------------------------------
        # Association counts
        # --------------------------------------------------

        matched_gt = set(
            current_gt_matches.keys()
        )


        for i, gt in enumerate(
            gt_objects
        ):

            identity = gt[
                "attributes"
            ]

            association_counts.setdefault(
                identity,
                {
                    "TPA": 0,
                    "FNA": 0
                }
            )

            if i not in matched_gt:

                association_counts[
                    identity
                ][
                    "FNA"
                ] += 1


        for i, j in (
            current_gt_matches.items()
        ):

            identity = gt_objects[
                i
            ][
                "attributes"
            ]

            association_counts[
                identity
            ][
                "TPA"
            ] += 1


        matched_predictions = set(
            current_gt_matches.values()
        )

        for j, pred in enumerate(
            pred_objects
        ):

            if j not in matched_predictions:

                identity = pred[
                    "attributes"
                ]

                false_positive_association_counts[
                    identity
                ] = (
                    false_positive_association_counts.get(
                        identity,
                        0
                    )
                    + 1
                )


    # Detection accuracy
    detection_denominator = (
        total_tp
        + total_fp
        + total_fn
    )

    det_a = (
        total_tp
        / detection_denominator
        if detection_denominator > 0
        else 0.0
    )


    # Association accuracy
    association_sum = 0.0
    n_identities = 0

    for identity, counts in (
        association_counts.items()
    ):

        denominator = (
            counts["TPA"]
            +
            counts["FNA"]
            +
            false_positive_association_counts.get(
                identity,
                0
            )
        )

        if denominator > 0:

            association_sum += (
                counts["TPA"]
                / denominator
            )

            n_identities += 1


    ass_a = (
        association_sum
        / n_identities
        if n_identities > 0
        else 0.0
    )


    ti_hota_alpha = math.sqrt(
        det_a * ass_a
    )


    return (
        ti_hota_alpha,
        det_a,
        ass_a,
        total_tp,
        total_fp,
        total_fn
    )


def compute_ti_hota(
    frames,
    tau
):

    alphas = np.arange(
        0.05,
        1.0,
        0.05
    )

    ti_hota_values = []
    det_a_values = []
    ass_a_values = []

    tp_values = []
    fp_values = []
    fn_values = []


    for alpha in alphas:

        (
            ti_hota_alpha,
            det_a,
            ass_a,
            tp,
            fp,
            fn
        ) = compute_ti_hota_alpha(
            frames,
            tau,
            alpha
        )

        ti_hota_values.append(
            ti_hota_alpha
        )

        det_a_values.append(
            det_a
        )

        ass_a_values.append(
            ass_a
        )

        tp_values.append(
            tp
        )

        fp_values.append(
            fp
        )

        fn_values.append(
            fn
        )


    metrics = {
        "TI-HOTA":
            float(
                np.mean(
                    ti_hota_values
                )
            ),

        "TI-HOTA_alphas":
            ti_hota_values,

        "TI-DetA":
            float(
                np.mean(
                    det_a_values
                )
            ),

        "TI-AssA":
            float(
                np.mean(
                    ass_a_values
                )
            ),

        "TI-TP":
            float(
                np.mean(
                    tp_values
                )
            ),

        "TI-FP":
            float(
                np.mean(
                    fp_values
                )
            ),

        "TI-FN":
            float(
                np.mean(
                    fn_values
                )
            ),
    }

    return (
        metrics,
        alphas
    )


# --------------------------------------------------
# Read Challenge-format files
# --------------------------------------------------

def load_gt_frames(file_path):
    """
    Read Challenge-format ground truth:

    frame_id, court_x, court_y, team_id, jersey_number

    team_id + jersey_number is the persistent
    semantic player identity.
    """

    gt_df = pd.read_csv(
        file_path
    )

    required_columns = [
        "frame_id",
        "court_x",
        "court_y",
        "team_id",
        "jersey_number",
    ]

    if list(gt_df.columns) != required_columns:
        raise ValueError(
            "GT columns must be:\n"
            + ", ".join(required_columns)
        )

    frames_dict = defaultdict(list)

    for row in gt_df.itertuples(index=False):

        semantic_identity = (
            str(row.team_id)
            + str(int(row.jersey_number))
        )

        frames_dict[
            int(row.frame_id)
        ].append({
            "position": [
                float(row.court_x),
                float(row.court_y)
            ],
            "attributes": semantic_identity
        })

    return frames_dict


def load_prediction_frames(predictions_df):
    """
    Convert the canonical final prediction table into
    the structure used by TI-HOTA.

    team_id + jersey_number is the persistent
    semantic player identity.
    """

    frames_dict = defaultdict(list)

    for row in predictions_df.itertuples(index=False):

        semantic_identity = (
            str(row.team_id)
            + str(int(row.jersey_number))
        )

        frames_dict[
            int(row.frame_id)
        ].append({
            "position": [
                float(row.court_x),
                float(row.court_y)
            ],
            "attributes": semantic_identity
        })

    return frames_dict


def load_video(
    gt_file,
    predictions_df
):

    gt_frames = load_gt_frames(
        gt_file
    )

    pred_frames = load_prediction_frames(
        predictions_df
    )

    all_frame_ids = sorted(
        set(gt_frames.keys())
        |
        set(pred_frames.keys())
    )

    frames = []

    for frame_id in all_frame_ids:

        frames.append({
            "ground_truths":
                gt_frames.get(
                    frame_id,
                    []
                ),

            "predictions":
                pred_frames.get(
                    frame_id,
                    []
                )
        })

    return frames


In [ ]:
# TrackID3x3 Kaggle cell: evaluation
with baseline_stage(11, 'Evaluation'):
    # ==============================
    # 9. Run TI-HOTA evaluation
    # ==============================

    print(
        "Ground truth:",
        gt_file
    )

    print(
        "Prediction  :",
        predictions_file
    )

    frames = load_video(
        gt_file,
        predictions_df
    )

    # Challenge coordinates are expressed in centimeters
    tau = 100.0

    metrics, alphas = compute_ti_hota(
        frames,
        tau
    )

    print(
        "\n=============================="
    )

    print(
        "TI-HOTA evaluation"
    )

    print(
        "=============================="
    )

    print(
        f"TI-HOTA: "
        f"{metrics['TI-HOTA']:.4f}"
    )

    print(
        f"TI-DetA: "
        f"{metrics['TI-DetA']:.4f}"
    )

    print(
        f"TI-AssA: "
        f"{metrics['TI-AssA']:.4f}"
    )

    print(
        f"TI-TP: "
        f"{metrics['TI-TP']:.2f}"
    )

    print(
        f"TI-FP: "
        f"{metrics['TI-FP']:.2f}"
    )

    print(
        f"TI-FN: "
        f"{metrics['TI-FN']:.2f}"
    )

    # ==============================
    # 9.1 Location-only diagnostics
    # ==============================

    def compute_location_diagnostics(
        gt_file,
        predictions_df,
        max_distance_cm=100.0
    ):
        """
        Match GT and predictions using court position only.

        Diagnostic metrics are computed only for matched pairs
        within max_distance_cm.
        """

        gt_df = pd.read_csv(gt_file)

        gt_by_frame = {
            int(frame_id): group.reset_index(drop=True)
            for frame_id, group in gt_df.groupby("frame_id")
        }

        pred_by_frame = {
            int(frame_id): group.reset_index(drop=True)
            for frame_id, group in predictions_df.groupby("frame_id")
        }

        position_errors = []
        team_correct = []
        jersey_correct = []

        all_frame_ids = sorted(
            set(gt_by_frame.keys())
            | set(pred_by_frame.keys())
        )

        for frame_id in all_frame_ids:

            gt = gt_by_frame.get(frame_id)
            pred = pred_by_frame.get(frame_id)

            if gt is None or pred is None:
                continue

            gt_xy = gt[
                ["court_x", "court_y"]
            ].to_numpy(dtype=float)

            pred_xy = pred[
                ["court_x", "court_y"]
            ].to_numpy(dtype=float)

            distances = np.linalg.norm(
                gt_xy[:, None, :]
                - pred_xy[None, :, :],
                axis=2
            )

            # Location-only Hungarian matching
            cost = distances.copy()
            cost[distances > max_distance_cm] = 1e6

            gt_indices, pred_indices = (
                linear_sum_assignment(cost)
            )

            for i, j in zip(
                gt_indices,
                pred_indices
            ):
                distance = distances[i, j]

                if distance > max_distance_cm:
                    continue

                position_errors.append(distance)

                team_correct.append(
                    str(gt.iloc[i]["team_id"])
                    == str(pred.iloc[j]["team_id"])
                )

                jersey_correct.append(
                    int(gt.iloc[i]["jersey_number"])
                    == int(pred.iloc[j]["jersey_number"])
                )

        if not position_errors:
            return {
                "matched_pairs": 0,
                "median_position_error_cm": np.nan,
                "team_accuracy": np.nan,
                "jersey_accuracy": np.nan,
            }

        return {
            "matched_pairs": len(position_errors),
            "median_position_error_cm":
                float(np.median(position_errors)),
            "team_accuracy":
                float(np.mean(team_correct)),
            "jersey_accuracy":
                float(np.mean(jersey_correct)),
        }


    diagnostics = compute_location_diagnostics(
        gt_file,
        predictions_df,
        max_distance_cm=100.0
    )

    print("\n==============================")
    print("Location-only diagnostics")
    print("==============================")

    print(
        "Matched pairs:",
        diagnostics["matched_pairs"]
    )

    print(
        "Median court-position error:",
        f"{diagnostics['median_position_error_cm']:.2f} cm"
    )

    print(
        "Team-identification accuracy:",
        f"{diagnostics['team_accuracy']:.4f}"
    )

    print(
        "Jersey-number accuracy:",
        f"{diagnostics['jersey_accuracy']:.4f}"
    )
    # OBSERVABILITY: persist the returned metrics; matching and in-memory precision are unchanged.
    save_json('ti_hota.json', dict(tau_cm=tau, alphas=alphas.tolist(), metrics=metrics))
    clean_diagnostics = {key: (None if isinstance(value, float) and not math.isfinite(value) else value)
                         for key,value in diagnostics.items()}
    save_json('location_diagnostics.json', clean_diagnostics)
    historical = {'TI-HOTA': 0.6351, 'TI-DetA': 0.6362, 'TI-AssA': 0.6341}
    save_json('historical_comparison.json', dict(reference='Unverified saved original-notebook output',
              historical=historical, fresh={k:metrics[k] for k in historical},
              fresh_minus_historical={k:metrics[k]-v for k,v in historical.items()},
              note='No exact-score equality requirement; seeds and algorithm remain unchanged.'))
    print('Persisted evaluation:', RUN_DIR / 'ti_hota.json')


## Optional visualization

When `SAVE_VIDEOS=True`, the following cell creates the original tracker video, rectified
court image, GT/pred minimaps, and combined video. It runs **after persisted evaluation**.
Only Colab display and destination paths are replaced. The original minimap time handling
and drawing parameters are preserved. No metric depends on this cell.


In [ ]:
# TrackID3x3 Kaggle cell: optional-videos
if SAVE_VIDEOS:
    with baseline_stage('optional', 'Visualizations'):
        saved_homography = H.copy()
        try:
            # Load MOT data for visualization
            # --------------------------------------------------

            mot_data = defaultdict(list)

            with open(mot_txt_path, "r") as f:

                for line in f:

                    (
                        frame_id,
                        track_id,
                        x,
                        y,
                        w,
                        h,
                        confidence,
                        *_
                    ) = line.strip().split(",")

                    mot_data[
                        int(frame_id)
                    ].append({
                        "id": int(track_id),
                        "bbox": (
                            float(x),
                            float(y),
                            float(w),
                            float(h)
                        ),
                        "conf": float(confidence)
                    })


            # --------------------------------------------------
            # Assign colors to ByteTrack IDs
            # --------------------------------------------------

            unique_ids = sorted({
                detection["id"]
                for detections in mot_data.values()
                for detection in detections
            })

            id_to_color = {}

            n_ids = max(
                len(unique_ids),
                1
            )

            for index, track_id in enumerate(unique_ids):

                hue = index / n_ids

                r, g, b = colorsys.hsv_to_rgb(
                    hue,
                    1.0,
                    1.0
                )

                id_to_color[track_id] = (
                    int(b * 255),
                    int(g * 255),
                    int(r * 255)
                )


            # --------------------------------------------------
            # Create tracking visualization
            # --------------------------------------------------

            output_tracking_video = os.path.join(
                out_dir,
                "tracked_visualization.mp4"
            )

            fourcc = cv2.VideoWriter_fourcc(
                *"mp4v"
            )

            cap = cv2.VideoCapture(
                video_path
            )

            writer = cv2.VideoWriter(
                output_tracking_video,
                fourcc,
                fps,
                (width, height)
            )

            while True:

                ret, frame = cap.read()

                if not ret:
                    break

                frame_id = int(
                    cap.get(
                        cv2.CAP_PROP_POS_FRAMES
                    )
                )

                for detection in mot_data.get(
                    frame_id,
                    []
                ):

                    track_id = detection["id"]

                    x, y, w, h = detection[
                        "bbox"
                    ]

                    color = id_to_color[
                        track_id
                    ]

                    cv2.rectangle(
                        frame,
                        (int(x), int(y)),
                        (int(x + w), int(y + h)),
                        color,
                        2
                    )

                    cv2.putText(
                        frame,
                        str(track_id),
                        (int(x), int(y) - 6),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.8,
                        color,
                        2
                    )

                writer.write(frame)

            cap.release()
            writer.release()

            print(
                "Tracking visualization:",
                output_tracking_video
            )

            # ==============================
            # Check the homography
            # ==============================

            from IPython.display import display, Image

            def cv2_imshow(image):
                ok, encoded = cv2.imencode(".png", image)
                if not ok:
                    raise RuntimeError("Could not encode court preview")
                display(Image(data=encoded.tobytes()))

            cap = cv2.VideoCapture(
                video_path
            )

            ret, first_frame = cap.read()

            cap.release()

            if not ret:

                raise IOError(
                    "Could not read the first video frame."
                )


            # 1 cm corresponds to 1 output pixel
            court_width_pixels = 1505
            court_height_pixels = 1105

            warped_court_image = cv2.warpPerspective(
                first_frame,
                H,
                (
                    court_width_pixels,
                    court_height_pixels
                )
            )

            # Resize only for display
            display_width = 1000

            display_height = int(
                display_width
                * court_height_pixels
                / court_width_pixels
            )

            display_image = cv2.resize(
                warped_court_image,
                (
                    display_width,
                    display_height
                )
            )

            cv2_imshow(
                display_image
            )

            output_image_path = (
                str(RUN_DIR) + "/"
                "transformed_court_image.jpg"
            )

            cv2.imwrite(
                output_image_path,
                warped_court_image
            )

            print(
                "Rectified court image:",
                output_image_path
            )

            # ==============================
            # 10. Minimap helper functions
            # ==============================

            import cv2
            import glob
            import os


            def load_gt_points_from_file(file_path):
                """
                Read Challenge-format GT data and return:
                (frame, x_cm, y_cm, attribute)
                """

                gt_df = pd.read_csv(
                    file_path
                )

                points = []

                for row in gt_df.itertuples(index=False):

                    frame = int(
                        row.frame_id
                    )

                    x = float(
                        row.court_x
                    )

                    y = float(
                        row.court_y
                    )

                    attr = (
                        str(row.team_id)
                        + str(int(row.jersey_number))
                    )

                    points.append(
                        (
                            frame,
                            x,
                            y,
                            attr
                        )
                    )

                return points


            def prediction_points_from_dataframe(
                predictions_df
            ):
                """
                Build minimap points from the same final prediction rows
                used for evaluation and video visualization.
                """

                points = []

                for row in predictions_df.itertuples(index=False):

                    points.append(
                        (
                            int(row.frame_id),
                            float(row.court_x),
                            float(row.court_y),
                            str(row.team_id)
                            + str(int(row.jersey_number))
                        )
                    )

                return points


            def scale_points_swapped(
                points,
                img_width,
                draw_height
            ):
                """
                Follow the original TrackID3x3 Outdoor minimap
                coordinate transformation.

                x is horizontally flipped.
                y is unchanged.
                """

                flipped_points = []

                for point in points:

                    frame, x, y, attr = point

                    flipped_x = (
                        img_width - x
                    )

                    flipped_points.append(
                        (
                            frame,
                            flipped_x,
                            y,
                            attr
                        )
                    )

                return flipped_points


            def get_color(attr):
                """
                Fixed Challenge team colors.

                A = initial Offense = purple
                B = initial Defense = green

                O/D are also accepted for compatibility.
                """

                if (
                    attr.startswith("A")
                    or
                    attr.startswith("O")
                ):
                    # purple
                    return (128, 0, 128)

                elif (
                    attr.startswith("B")
                    or
                    attr.startswith("D")
                ):
                    # green
                    return (0, 200, 0)

                else:
                    return (0, 0, 0)


            def build_attribute_color_mapping_from_file(
                gt_file
            ):
                """
                Build attribute -> color mapping from GT.
                """

                mapping = {}

                pts = load_gt_points_from_file(
                    gt_file
                )

                for p in pts:

                    _, _, _, attr = p

                    if attr not in mapping:

                        mapping[attr] = get_color(
                            attr
                        )

                return mapping


            def draw_points(
                image,
                points
            ):
                """
                Draw GT points using the original
                create_minimap.ipynb drawing parameters.
                """

                for point in points:

                    frame, x, y, attr = point

                    color = get_color(
                        attr
                    )

                    x_int = int(x)
                    y_int = int(y)

                    cv2.circle(
                        image,
                        (x_int, y_int),
                        radius=30,
                        color=color,
                        thickness=-1
                    )

                    cv2.putText(
                        image,
                        attr,
                        (
                            x_int + 10,
                            y_int - 10
                        ),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        4,
                        color,
                        thickness=8
                    )

                return image


            def draw_points_with_mapping(
                image,
                points,
                attr_color_mapping
            ):
                """
                Draw prediction points using the GT
                attribute-color mapping.
                """

                for point in points:

                    frame, x, y, attr = point

                    if attr in attr_color_mapping:

                        color = (
                            attr_color_mapping[
                                attr
                            ]
                        )

                    else:

                        color = get_color(
                            attr
                        )

                    x_int = int(x)
                    y_int = int(y)

                    cv2.circle(
                        image,
                        (x_int, y_int),
                        radius=30,
                        color=color,
                        thickness=-1
                    )

                    cv2.putText(
                        image,
                        attr,
                        (
                            x_int + 10,
                            y_int - 10
                        ),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        4,
                        color,
                        thickness=8
                    )

                return image

            # ==============================
            # 11. Create GT and prediction minimaps
            # ==============================

            # Common settings
            base_name = os.path.splitext(
                os.path.basename(
                    video_path
                )
            )[0]


            # --------------------------------------------------
            # Load and resize Outdoor.png
            # --------------------------------------------------

            base_image = cv2.imread(
                court_image_path
            )

            if base_image is None:

                raise FileNotFoundError(
                    f"Could not read court image: "
                    f"{court_image_path}"
                )


            # Original TrackID3x3 Outdoor minimap dimensions
            resized_image = cv2.resize(
                base_image,
                (1505, 1105)
            )

            height, width = (
                resized_image.shape[:2]
            )

            print(
                f"Image size: "
                f"width={width}, "
                f"height={height}, "
                f"drawing area={height}px"
            )


            # --------------------------------------------------
            # Ground Truth minimap
            # --------------------------------------------------

            pts = load_gt_points_from_file(
                gt_file
            )

            if pts:

                scaled = scale_points_swapped(
                    pts,
                    width,
                    height
                )

                frames = {}

                for f, *data in scaled:

                    frames.setdefault(
                        f,
                        []
                    ).append(
                        [f, *data]
                    )


                cap = cv2.VideoCapture(
                    video_path
                )

                fps = (
                    cap.get(
                        cv2.CAP_PROP_FPS
                    )
                    or 10
                )

                cap.release()

                print(
                    f"GT Using FPS: {fps}"
                )


                gt_minimap_path = os.path.join(
                    str(RUN_DIR),
                    f"{base_name}_GT.mp4"
                )

                vw = cv2.VideoWriter(
                    gt_minimap_path,
                    cv2.VideoWriter_fourcc(
                        *"mp4v"
                    ),
                    fps,
                    (
                        width,
                        height
                    )
                )


                for f in sorted(
                    frames
                ):

                    img = (
                        resized_image.copy()
                    )

                    img = draw_points(
                        img,
                        frames[f]
                    )

                    vw.write(
                        img
                    )


                vw.release()

                print(
                    f"GT video saved: "
                    f"{gt_minimap_path}"
                )


            # --------------------------------------------------
            # Prediction minimap
            # --------------------------------------------------

            pts = prediction_points_from_dataframe(
                predictions_df
            )

            if pts:

                scaled = scale_points_swapped(
                    pts,
                    width,
                    height
                )

                frames = {}

                for f, *data in scaled:

                    frames.setdefault(
                        f,
                        []
                    ).append(
                        [f, *data]
                    )


                mapping = (
                    build_attribute_color_mapping_from_file(
                        gt_file
                    )
                )


                cap = cv2.VideoCapture(
                    video_path
                )

                fps = (
                    cap.get(
                        cv2.CAP_PROP_FPS
                    )
                    or 10
                )

                cap.release()

                print(
                    f"Prediction Using FPS: {fps}"
                )


                pred_minimap_path = os.path.join(
                    str(RUN_DIR),
                    f"{base_name}_pred.mp4"
                )

                vw = cv2.VideoWriter(
                    pred_minimap_path,
                    cv2.VideoWriter_fourcc(
                        *"mp4v"
                    ),
                    fps,
                    (
                        width,
                        height
                    )
                )


                for f in sorted(
                    frames
                ):

                    img = (
                        resized_image.copy()
                    )

                    img = draw_points_with_mapping(
                        img,
                        frames[f],
                        mapping
                    )

                    vw.write(
                        img
                    )


                vw.release()

                print(
                    f"Prediction video saved: "
                    f"{pred_minimap_path}"
                )


            # ==============================
            # 12. Overlay minimaps
            # ==============================

            import os
            import cv2
            import numpy as np


            # --------------------------------------------------
            # File paths
            # --------------------------------------------------

            input_video_path = video_path

            pred_video_path = pred_minimap_path
            gt_video_path = gt_minimap_path

            output_video_path = os.path.join(
                str(RUN_DIR),
                f"{base_name}_with_minimap.mp4"
            )


            # --------------------------------------------------
            # Use the final prediction table for video drawing
            # --------------------------------------------------

            prediction_rows_by_frame = {
                int(frame_id): group.copy()
                for frame_id, group
                in predictions_df.groupby("frame_id")
            }


            # --------------------------------------------------
            # Open videos
            # --------------------------------------------------

            cap = cv2.VideoCapture(
                input_video_path
            )

            pred_cap = cv2.VideoCapture(
                pred_video_path
            )

            gt_cap = cv2.VideoCapture(
                gt_video_path
            )

            if not cap.isOpened():

                raise RuntimeError(
                    f"Could not open video: "
                    f"{input_video_path}"
                )

            main_width = int(
                cap.get(
                    cv2.CAP_PROP_FRAME_WIDTH
                )
            )

            main_height = int(
                cap.get(
                    cv2.CAP_PROP_FRAME_HEIGHT
                )
            )

            fps = (
                cap.get(
                    cv2.CAP_PROP_FPS
                )
                or 10
            )

            fourcc = (
                cv2.VideoWriter_fourcc(
                    *"mp4v"
                )
            )

            out = cv2.VideoWriter(
                output_video_path,
                fourcc,
                fps,
                (
                    main_width,
                    main_height
                )
            )


            # --------------------------------------------------
            # Original TrackID3x3 minimap sizing
            # --------------------------------------------------

            M_base = (
                main_width / 5.0
            )

            L = (
                4 / 3.0
            ) * M_base

            # Read first prediction minimap frame
            ret_pred, first_pred = (
                pred_cap.read()
            )

            if not ret_pred:

                raise RuntimeError(
                    "Failed to read prediction minimap."
                )

            orig_pred_h, orig_pred_w = (
                first_pred.shape[:2]
            )

            r = (
                orig_pred_h
                / orig_pred_w
            )

            # Original code enlarges the minimap by 1.1
            M = (
                M_base * 1.1
            )

            H = (
                M * r
            )

            # Reset minimap frame positions
            pred_cap.set(
                cv2.CAP_PROP_POS_FRAMES,
                0
            )

            gt_cap.set(
                cv2.CAP_PROP_POS_FRAMES,
                0
            )

            # Original placement
            label_space = 20

            overlay_y = int(
                label_space
            )

            pred_x = int(
                L
            )

            gt_x = int(
                L
                + M_base
                + (
                    M_base / 3.0
                )
            )


            # --------------------------------------------------
            # Label settings
            # --------------------------------------------------

            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 2
            font_thickness = 5
            label_color = (
                255,
                255,
                255
            )


            # --------------------------------------------------
            # Process video
            # --------------------------------------------------

            frame_index = 1

            while True:

                ret_main, main_frame = (
                    cap.read()
                )

                if not ret_main:
                    break

                current_frame = frame_index


                # --------------------------------------------------
                # Draw the same final predictions on the original video
                # --------------------------------------------------

                if current_frame in prediction_rows_by_frame:

                    frame_predictions = prediction_rows_by_frame[
                        current_frame
                    ]

                    for row in frame_predictions.itertuples(index=False):

                        x = float(row.bbox_x)
                        y = float(row.bbox_y)
                        w = float(row.bbox_width)
                        h = float(row.bbox_height)

                        if not np.isfinite(
                            [x, y, w, h]
                        ).all():
                            continue

                        a = w / 2
                        b = a / 5

                        if (
                            a <= 1
                            or
                            b <= 1
                        ):
                            continue

                        cx = int(
                            x + w / 2
                        )

                        cy = int(
                            y
                            + h
                            + b / 2
                        )

                        attribute = (
                            str(row.team_id)
                            + str(int(row.jersey_number))
                        )

                        color = get_color(
                            attribute
                        )

                        if color == (
                            0,
                            0,
                            0
                        ):
                            continue

                        cv2.ellipse(
                            main_frame,
                            (
                                cx,
                                cy
                            ),
                            (
                                int(a),
                                int(b)
                            ),
                            0,
                            330,
                            570,
                            color,
                            3
                        )

                        text_pos = (
                            int(x + 25),
                            int(y - 10)
                        )

                        cv2.putText(
                            main_frame,
                            attribute,
                            text_pos,
                            cv2.FONT_HERSHEY_SIMPLEX,
                            1.3,
                            color,
                            3,
                            cv2.LINE_AA
                        )


                # --------------------------------------------------
                # Overlay minimaps at the top
                # --------------------------------------------------

                ret_pred, frame_pred = (
                    pred_cap.read()
                )

                ret_gt, frame_gt = (
                    gt_cap.read()
                )

                if (
                    ret_pred
                    and
                    ret_gt
                ):

                    frame_pred_resized = (
                        cv2.resize(
                            frame_pred,
                            (
                                int(M),
                                int(H)
                            ),
                            interpolation=(
                                cv2.INTER_LINEAR
                            )
                        )
                    )

                    frame_gt_resized = (
                        cv2.resize(
                            frame_gt,
                            (
                                int(M),
                                int(H)
                            ),
                            interpolation=(
                                cv2.INTER_LINEAR
                            )
                        )
                    )

                    alpha = 0.9

                    # Prediction minimap
                    roi_pred = main_frame[
                        overlay_y:
                            overlay_y + int(H),
                        pred_x:
                            pred_x + int(M)
                    ]

                    main_frame[
                        overlay_y:
                            overlay_y + int(H),
                        pred_x:
                            pred_x + int(M)
                    ] = cv2.addWeighted(
                        roi_pred,
                        1 - alpha,
                        frame_pred_resized,
                        alpha,
                        0
                    )

                    # Ground Truth minimap
                    roi_gt = main_frame[
                        overlay_y:
                            overlay_y + int(H),
                        gt_x:
                            gt_x + int(M)
                    ]

                    main_frame[
                        overlay_y:
                            overlay_y + int(H),
                        gt_x:
                            gt_x + int(M)
                    ] = cv2.addWeighted(
                        roi_gt,
                        1 - alpha,
                        frame_gt_resized,
                        alpha,
                        0
                    )

                    # --------------------------------------------------
                    # Labels
                    # --------------------------------------------------

                    pred_label_x = (
                        pred_x
                        + int(M / 2)
                    )

                    gt_label_x = (
                        gt_x
                        + int(M / 2)
                    )

                    label_y = (
                        overlay_y
                        + int(H)
                        + label_space
                        + 42
                    )

                    # Same geometry as the English fallback
                    # in the original draw_minimap.ipynb.
                    cv2.putText(
                        main_frame,
                        "Prediction",
                        (
                            pred_label_x - 150,
                            label_y
                        ),
                        font,
                        font_scale,
                        label_color,
                        font_thickness
                    )

                    cv2.putText(
                        main_frame,
                        "Ground Truth",
                        (
                            gt_label_x - 195,
                            label_y
                        ),
                        font,
                        font_scale,
                        label_color,
                        font_thickness
                    )

                out.write(
                    main_frame
                )

                frame_index += 1


            # --------------------------------------------------
            # Release resources
            # --------------------------------------------------

            cap.release()
            pred_cap.release()
            gt_cap.release()
            out.release()

            print(
                "Processing is complete. "
                f"Output: {output_video_path}"
            )
        finally:
            H = saved_homography
else:
    print('Optional videos skipped (SAVE_VIDEOS=False); predictions and metrics are already saved.')


In [ ]:
# TrackID3x3 Kaggle cell: finish
with baseline_stage('finalize', 'Evidence'):
    capture_notebook_source(get_ipython().history_manager.input_hist_raw)
    save_json('model_hashes.json', {str(p.relative_to(RUN_DIR)): file_sha256(p)
              for p in sorted((RUN_DIR / 'models').rglob('*')) if p.is_file()})
    save_json('output_summary.json', dict(predictions_path=predictions_file, prediction_rows=len(predictions_df),
              prediction_frames=int(predictions_df.frame_id.nunique()), predictions_sha256=file_sha256(predictions_file),
              metric_precision='original in-memory predictions', metrics_path=str(RUN_DIR / 'ti_hota.json'),
              original_notebook_sha256=ORIGINAL_NOTEBOOK_SHA256))
    RUN_STATE['status'] = 'PASS' if IS_KAGGLE else 'COMPLETED_ON_OTHER_CUDA_HOST'
    print('Metric pipeline completed. Outputs:', RUN_DIR)
save_json('artifact_hashes.json', {str(p.relative_to(RUN_DIR)): file_sha256(p)
          for p in sorted(RUN_DIR.rglob('*')) if p.is_file() and p.name != 'artifact_hashes.json'})
